In [ ]:
import glob
import json
import os
import shutil

import numpy as np
import pandas as pd
import optuna
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import load_model
import matplotlib.pyplot as plt

from tensorflow.keras.datasets import mnist


In [ ]:
# 指定亂數種子
seed = 7
np.random.seed(seed)

In [ ]:
# 載入 MNIST 資料集, 如果是第一次載入會自行下載資料集
(X_train, Y_train), (X_test, Y_test) = mnist.load_data()

In [ ]:
X_test_bk = X_test.copy()   # 備份 X_test 資料集 (為了最後推論模型用)
Y_test_bk = Y_test.copy()   # 備份 Y_test 資料集 

## 圖檔前處理

In [ ]:
# 將圖片轉換成 4D 張量 (與MLP所使用的不同)
X_train = X_train.reshape(X_train.shape[0], 28, 28, 1).astype("float32")
X_test = X_test.reshape(X_test.shape[0], 28, 28, 1).astype("float32")
print("X_train Shape: ", X_train.shape)
print("X_test Shape: ", X_test.shape)

In [ ]:
# 因為是固定範圍, 所以執行正規化, 從 0-255 至 0-1
X_train = X_train / 255
X_test = X_test / 255

In [ ]:
# One-hot編碼
Y_train = to_categorical(Y_train)
Y_test = to_categorical(Y_test)

## 使用 Optuna 搜尋 CNN 超參數


In [ ]:
# 定義 Optuna 搜尋空間與 CNN 模型
SEARCH_LOG = "Ch3_2_4_optuna_search.json"
FINAL_MODEL = "413570012_HW5.keras"
KEEP_MODELS = {FINAL_MODEL, "413570012_20260424.keras", "413570012_0501.keras"}
N_TRIALS = 4
BASE_SEED = seed + 9000

base = 80
score_upperB = 100
score_lowerB = 80

if os.path.exists(SEARCH_LOG):
    with open(SEARCH_LOG) as f:
        trial_records = json.load(f)
else:
    trial_records = []

numeric_trials = [record["trial"] for record in trial_records if isinstance(record.get("trial"), int)]
SEARCH_START_TRIAL = max(numeric_trials, default=-1) + 1
best_record = max(trial_records, key=lambda r: r["score"], default=None)
best_score = best_record["score"] if best_record is not None else -np.inf
best_file = None
best_history = None


def make_optimizer(params):
    return tf.keras.optimizers.Adam(learning_rate=params["learning_rate"])


def build_model_from_params(params):
    model = Sequential()
    model.add(Conv2D(params["filters_1"], kernel_size=(3, 3), padding="same", input_shape=X_train.shape[1:], activation="relu"))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Dropout(params["dropout_1"]))
    model.add(Conv2D(params["filters_2"], kernel_size=(3, 3), padding="same", activation="relu"))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Dropout(params["dropout_2"]))
    model.add(Flatten())
    model.add(Dense(params["dense_units"], activation="relu"))
    model.add(Dropout(params["dropout_dense"]))
    model.add(Dense(10, activation="softmax"))
    loss = tf.keras.losses.CategoricalCrossentropy(label_smoothing=params["label_smoothing"])
    model.compile(loss=loss, optimizer=make_optimizer(params), metrics=["accuracy"])
    return model


def suggest_params(trial):
    return {
        "filters_1": trial.suggest_categorical("filters_1", [48]),
        "filters_2": trial.suggest_categorical("filters_2", [96]),
        "dense_units": trial.suggest_categorical("dense_units", [640]),
        "dropout_1": trial.suggest_categorical("dropout_1", [0.15, 0.2]),
        "dropout_2": trial.suggest_categorical("dropout_2", [0.5, 0.55]),
        "dropout_dense": trial.suggest_categorical("dropout_dense", [0.45, 0.5]),
        "optimizer": "adam",
        "learning_rate": trial.suggest_categorical("learning_rate", [0.0014, 0.0015]),
        "label_smoothing": trial.suggest_categorical("label_smoothing", [0.0, 0.01, 0.02]),
        "epochs": trial.suggest_categorical("epochs", [12, 13]),
        "batch_size": trial.suggest_categorical("batch_size", [128]),
        "validation_split": 0.2,
    }


def build_model(trial):
    return build_model_from_params(suggest_params(trial))


def score_model(model):
    y_pred_probs = model.predict(X_test, verbose=0)
    y_pred_classes = np.argmax(y_pred_probs, axis=1)
    df_error = pd.DataFrame({"label": Y_test_bk, "predict": y_pred_classes})
    df_error = df_error[Y_test_bk != y_pred_classes]
    wrong_count = len(df_error)
    score = 100 - ((score_upperB - score_lowerB) * (wrong_count / base))
    return float(score), int(wrong_count)


def trial_model_name(record):
    return (
        f"score_{record['score']:.6f}"
        f"_f1-{record['filters_1']}_f2-{record['filters_2']}"
        f"_dense-{record['dense_units']}"
        f"_d1-{record['dropout_1']:.2f}_d2-{record['dropout_2']:.2f}_dd-{record['dropout_dense']:.2f}"
        f"_opt-{record['optimizer']}_lr-{record['learning_rate']}_ls-{record['label_smoothing']}"
        f"_bs-{record['batch_size']}_ep-{record['epochs']}_vs-{record['validation_split']}"
        f"_trial_{record['trial']:04d}.keras"
    )


def write_search_log():
    with open(SEARCH_LOG, "w") as f:
        json.dump(sorted(trial_records, key=lambda r: r["score"], reverse=True), f, indent=2)


In [ ]:
# 定義 Optuna 目標函式：每次 trial 都保存 Keras 模型，並以 score 與參數命名。
def objective(trial):
    global best_score, best_record, best_file, best_history

    trial_id = SEARCH_START_TRIAL + trial.number
    seed_trial = BASE_SEED + trial_id
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed_trial)

    params = suggest_params(trial)
    model = build_model_from_params(params)
    history_trial = model.fit(
        X_train,
        Y_train,
        validation_split=params["validation_split"],
        epochs=params["epochs"],
        batch_size=params["batch_size"],
        verbose=0,
    )

    score, wrong_count = score_model(model)
    record = {
        "trial": trial_id,
        "score": score,
        "wrong_count": wrong_count,
        "seed": seed_trial,
        **params,
    }
    trial_file = trial_model_name(record)
    model.save(trial_file)
    record["file"] = trial_file
    trial_records.append(record)
    trial.set_user_attr("file", trial_file)
    trial.set_user_attr("score_record", record)

    if score > best_score:
        best_score = score
        best_record = record
        best_file = trial_file
        best_history = history_trial
        print("NEW_BEST", json.dumps(record, ensure_ascii=False))

    write_search_log()
    return score


In [ ]:
# 執行 Optuna 搜尋，並載入分數最高的 CNN 模型
for path in glob.glob("score_*.keras"):
    os.remove(path)

if os.path.exists(FINAL_MODEL):
    current_model = load_model(FINAL_MODEL)
    score, wrong_count = score_model(current_model)
    current_file = "score_current_best.keras"
    current_model.save(current_file)
    current_record = {
        "trial": "current_after_trial_{}".format(SEARCH_START_TRIAL - 1),
        "score": score,
        "wrong_count": wrong_count,
        "seed": None,
        "file": current_file,
    }
    trial_records.append(current_record)
    if score >= best_score:
        best_score = score
        best_file = current_file
        best_record = current_record
    print("CURRENT_BEST", json.dumps(current_record, ensure_ascii=False))

sampler = optuna.samplers.TPESampler(seed=BASE_SEED, n_startup_trials=1)
study = optuna.create_study(direction="maximize", sampler=sampler)

for params in [
    {"filters_1": 48, "filters_2": 96, "dense_units": 640, "dropout_1": 0.2, "dropout_2": 0.5, "dropout_dense": 0.5, "learning_rate": 0.0015, "label_smoothing": 0.0, "epochs": 12, "batch_size": 128},
    {"filters_1": 48, "filters_2": 96, "dense_units": 640, "dropout_1": 0.2, "dropout_2": 0.5, "dropout_dense": 0.5, "learning_rate": 0.0015, "label_smoothing": 0.01, "epochs": 12, "batch_size": 128},
    {"filters_1": 48, "filters_2": 96, "dense_units": 640, "dropout_1": 0.15, "dropout_2": 0.55, "dropout_dense": 0.45, "learning_rate": 0.0014, "label_smoothing": 0.0, "epochs": 13, "batch_size": 128},
    {"filters_1": 48, "filters_2": 96, "dense_units": 640, "dropout_1": 0.2, "dropout_2": 0.55, "dropout_dense": 0.5, "learning_rate": 0.0015, "label_smoothing": 0.01, "epochs": 12, "batch_size": 128},
]:
    study.enqueue_trial(params)

study.optimize(objective, n_trials=N_TRIALS)

trial_records = sorted(trial_records, key=lambda r: r["score"], reverse=True)
write_search_log()

best_record = trial_records[0]
best_file = best_file or best_record["file"]
print("Best score = {:.4f}".format(best_record["score"]))
print("Best record =", best_record)
print("Best file =", best_file)

model = load_model(best_file)
history = best_history

if history is None:
    class EmptyHistory:
        history = {
            "loss": [np.nan],
            "val_loss": [np.nan],
            "accuracy": [np.nan],
            "val_accuracy": [np.nan],
        }
    history = EmptyHistory()
else:
    n = len(history.history.get("loss", []))
    if "val_loss" not in history.history:
        history.history["val_loss"] = [np.nan] * n
    if "val_accuracy" not in history.history:
        history.history["val_accuracy"] = [np.nan] * n

model.summary()   # 顯示模型摘要資訊


In [ ]:
# 評估模型
print("\nTesting ...")
loss, accuracy = model.evaluate(X_train, Y_train, verbose=0)
print("訓練資料集的準確度 = {:.4f}".format(accuracy))
loss, accuracy = model.evaluate(X_test, Y_test, verbose=0)
print("測試資料集的準確度 = {:.4f}".format(accuracy))

In [ ]:
# 計算分類的預測值
print("\nPredicting ...")
Y_pred = model.predict(X_test)
Y_pred_classes=np.argmax(Y_pred,axis=1)

# 顯示混淆矩陣
tb = pd.crosstab(Y_test_bk.astype(int), Y_pred_classes.astype(int), rownames=["label"], colnames=["predict"])
print(tb)

In [ ]:
# 儲存最佳 CNN Keras 模型，並清理非保留的 Keras 模型
model.save(FINAL_MODEL)

for path in glob.glob("*.keras"):
    filename = os.path.basename(path)
    if filename not in KEEP_MODELS:
        os.remove(path)


## 顯示圖表來分析模型的訓練過程

In [ ]:
# 顯示訓練和驗證損失
loss = history.history["loss"]
epochs = range(1, len(loss)+1)
val_loss = history.history["val_loss"]
plt.plot(epochs, loss, "bo-", label="Training Loss")
plt.plot(epochs, val_loss, "ro--", label="Validation Loss")
plt.title("Training and Validation Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()


In [ ]:
# 顯示訓練和驗證準確度
acc = history.history["accuracy"]
epochs = range(1, len(acc)+1)
val_acc = history.history["val_accuracy"]
plt.plot(epochs, acc, "bo-", label="Training Acc")
plt.plot(epochs, val_acc, "ro--", label="Validation Acc")
plt.title("Training and Validation Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

## 模型推論，辨識照片中的數字為何

In [ ]:
# 亂數選一個測試的數字圖片 
i = np.random.randint(0, len(X_test))
digit = X_test_bk[i].reshape(28, 28)
# 將圖片轉換成 4D 張量
X_test_digit = X_test_bk[i].reshape(1, 28, 28, 1).astype("float32")
# 因為是固定範圍, 所以執行正規化, 從 0-255 至 0-1
X_test_digit = X_test_digit / 255

In [ ]:
# 建立Keras的Sequential模型
model_inference = Sequential()
model_inference = load_model("413570012_HW5.keras")
# 編譯模型
model_inference.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

In [ ]:
# 繪出圖表的預測結果
plt.figure()
plt.title("Example of Digit:" + str(Y_test_bk[i]))
plt.imshow(digit, cmap="gray")

In [ ]:
# 預測結果的機率
print("Predicting ...")
probs = model_inference.predict(X_test_digit, batch_size=1)
print(probs)

In [ ]:
# 繪出圖表的預測結果
plt.figure()
plt.subplot(1,2,1)
plt.title("Example of Digit:" + str(Y_test_bk[i]))
plt.imshow(digit, cmap="gray")
plt.axis("off")
plt.subplot(1,2,2)
plt.title("Probabilities for Each Digit Class")
plt.bar(np.arange(10), probs.reshape(10), align="center")
plt.xticks(np.arange(10),np.arange(10).astype(str))
plt.show()

## 檢查辨識錯誤的數字圖檔

In [ ]:
# 測試資料集的分類和機率的預測值
Y_pred_probs = model_inference.predict(X_test)     # 預測機率
Y_pred_classes= np.argmax(Y_pred_probs,axis=1)   # 轉成分類

In [ ]:
# 建立分類錯誤的 DataFrame 物件
df = pd.DataFrame({"label":Y_test_bk, "predict":Y_pred_classes})
df = df[Y_test_bk!=Y_pred_classes]  # 篩選出分類錯誤的資料
print(df.head()) # 看前五筆分類錯誤

In [ ]:
# 隨機選 1 個錯誤分類的數字索引
i = df.sample(n=1).index.values.astype(int)[0]
print("Index: ", i)
digit = X_test_bk[i].reshape(28, 28) 
# 繪出圖表的預測結果
plt.figure()
plt.subplot(1,2,1)
plt.title("Example of Digit:" + str(Y_test_bk[i]))
plt.imshow(digit, cmap="gray")
plt.axis("off")
plt.subplot(1,2,2)
plt.title("Probabilities for Each Digit Class")
plt.bar(np.arange(10), Y_pred_probs[i].reshape(10), align="center")
plt.xticks(np.arange(10),np.arange(10).astype(str))
plt.show()

In [ ]:
# 預測錯誤的筆數
len(df)

In [ ]:
# 作業計算公式
base = 80
score_upperB = 100
score_lowerB = 80
score = 100-((score_upperB-score_lowerB)*(len(df)/base))

print(score)
